# Data Preparation

This notebook loads from `data/processed/01_raw_loaded.parquet` produced by `01_ingest.ipynb` and will output a cleaned and transformed dataframe ready to be modelled to `data/processed/03_prep_clean.parquet` for use by `04_features.ipynb`. The key preparation actions are:
1. Nulls analysis
2. Categorical Variable encoding
3. Column renaming

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

sys.path.append(".")

df = pd.read_parquet(Path("../data/processed/01_raw_loaded.parquet"))

In [2]:
GRADE_COL = "aptem__UserILRSummary_GradingOutcome"
LEVEL_COL = "Apprenticeship Level"
EPAO_COL = "EPAO"
AGE_COL = "Age Bracket"
DATE_COL = "Achievement Date (ILR)"
PROGRAMME_COL = "Current Programme (Aptem)"

## Null Analysis

We need to check whether there is any bias that is being introduced by null values and if the dataset is imbalanced such that a certain level has more nulls than others. If nulls are randomly distributed across levels and EPAOs, we can safely remove them without introducing bias. If they are concentrated in a particular group, removing them could underrepresent that group in the model, leading to biased predictions.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 335 entries, 0 to 334
Data columns (total 6 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   Current Programme (Aptem)             335 non-null    object        
 1   Apprenticeship Level                  335 non-null    int64         
 2   Achievement Date (ILR)                335 non-null    datetime64[ns]
 3   EPAO                                  335 non-null    object        
 4   aptem__UserILRSummary_GradingOutcome  318 non-null    object        
 5   Age Bracket                           329 non-null    object        
dtypes: datetime64[ns](1), int64(1), object(4)
memory usage: 15.8+ KB


In [4]:
df[df[GRADE_COL].isnull()].groupby(LEVEL_COL).size()

Apprenticeship Level
3     5
4    10
7     2
dtype: int64

In [5]:
df[df[AGE_COL].isnull()].groupby(LEVEL_COL).size()

Apprenticeship Level
3    3
4    3
dtype: int64

### Observation

The distribution shows that the grade column has a slight imbalance with level 4 students having ~60% of nulls compared to the other grades whilst the age column has a more even split of nulls between level 3 and 4. Given the relatively small number of nulls in each level I believe that removing them would not drastically imbalance the dataset and won't bias the model given that we are startifying this by level anyways. Removing 17 grade outcome nulls (5.1% of 335 records) and 6 age bracket nulls will have minimal impact on the dataset.

In [6]:
df_clean = df.dropna(subset=[GRADE_COL,AGE_COL])
print(f"Rows before: {len(df)}")
print(f"Rows after: {len(df_clean)}")
print(f"Nulls remaining:\n{df_clean[[GRADE_COL, AGE_COL]].isnull().sum()}")

Rows before: 335
Rows after: 312
Nulls remaining:
aptem__UserILRSummary_GradingOutcome    0
Age Bracket                             0
dtype: int64


## Rename Columns

In [7]:
df_clean = df_clean.rename(columns={
    'aptem__UserILRSummary_GradingOutcome': 'grade',
    'Apprenticeship Level': 'level',
    'EPAO' : 'epao',
    'Age Bracket': 'age_bracket',
    'Achievement Date (ILR)': 'completion_date',
    'Current Programme (Aptem)': 'current_prog'
})

In [8]:
print(df_clean.columns.tolist())

['current_prog', 'level', 'completion_date', 'epao', 'grade', 'age_bracket']


In [9]:
GRADE_COL = "grade"
LEVEL_COL = "level"
EPAO_COL = "epao"
AGE_COL = "age_bracket"
DATE_COL = "completion_date"
PROGRAMME_COL = "current_prog"

## Encoding

I initially consider that I would ordinally encode level but given I decided to stratify the data by level and build separate models for each level I will not do this and only encode the other categorical variables like epao, age_bracket and grade. Level will act as a label. 

In [10]:
df_clean['is_distinction'] = np.where(df_clean[GRADE_COL]=='Distinction',1,0)

In [11]:
df_clean = pd.get_dummies(df_clean, columns=[EPAO_COL, AGE_COL], drop_first=True, dtype=int)

In [12]:
df_clean

,current_prog,level,completion_date,grade,is_distinction,epao_AIM,epao_AP,epao_BCS,age_bracket_30-49,age_bracket_50+
0,L3 NHS Data Citizen Apprenticeship Standard,3,2023-03-06,Pass,0,0,1,0,0,1
1,L7 AI Data Specialist Delivery Programme [L7 D...,4,2023-03-07,Pass,0,0,0,1,0,0
2,L3 NHS Data Citizen Apprenticeship Standard,3,2023-03-08,Pass,0,0,1,0,1,0
4,L7 AI Data Specialist Delivery Programme [L7 D...,7,2023-03-08,Pass,0,0,0,1,1,0
7,Data Analyst L4 Apprenticeship Standard (Sept ...,4,2023-03-13,Pass,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...
329,L7 AI Data Specialist Delivery Programme (Sept...,7,2024-02-26,Pass,0,0,0,1,0,0
331,L3 2.0 NHS Sept 2022 Cohort,3,2024-02-28,Distinction,1,0,1,0,0,0
332,Data Analyst L4 Apprenticeship Standard (Cohor...,4,2024-02-29,Distinction,1,0,1,0,1,0
333,Data Analyst L4 Apprenticeship Standard (Cohor...,4,2024-02-29,Distinction,1,0,1,0,0,0


In [13]:
# Write to Parquet file

processed_data_path = Path("../data/processed/03_prep_clean.parquet")
df_clean.to_parquet(processed_data_path,index=False)

## Final

This notebook prepped the data for modelling by removing nulls, renaming columns meaningfully and encoding categorical variables. 
The prepped data is exported to `data/processed/03_prep_clean.parquet`
The notebook `04_features.ipynb` will pick this data up